# Actividad: carga, transformación y resumen de un dataset de Hugging Face

Flujo del notebook:

```
HFDatasetLoader → DataFrame → transformación mínima → DatasetSummarizer → archivo .txt
```

- **Dataset:** [PixelPulse64/canadian_addresses](https://huggingface.co/datasets/PixelPulse64/canadian_addresses) (direcciones de Canadá, split `train`, unos 10 millones de filas).
- `hf_dataset_loader.py` contiene la clase que se conecta a Hugging Face y carga los datos.
- `dataset_summarizer.py` contiene la clase que genera el resumen en `.txt`.
- Este notebook coordina el flujo y realiza la transformación.

## 1. Importar las clases

In [1]:
from hf_dataset_loader import HFDatasetLoader
from dataset_summarizer import DatasetSummarizer

## 2. Obtener el DataFrame desde Hugging Face

El dataset completo tiene más de 10 millones de filas, así que se cargan solo las primeras 50.000 con `max_rows` (la clase usa *streaming* y no descarga el dataset completo).

In [2]:
loader = HFDatasetLoader("PixelPulse64/canadian_addresses", split="train", max_rows=50000)
df = loader.load()

print("Dimensiones:", df.shape)
df.head()

README.md:   0%|          | 0.00/523 [00:00<?, ?B/s]

Dimensiones: (50000, 7)


,unit_number,street_number,street,city,province,postal_code,__index_level_0__
0,NaN,410,HALL CR,YELLOW KNIFE,NT,None,0
1,NaN,121,NIVEN DR,YELLOW KNIFE,NT,None,1
2,NaN,5608,51A AVE,YELLOW KNIFE,NT,None,2
3,NaN,233,BORDEN DR,YELLOW KNIFE,NT,None,3
4,NaN,604,WILLIAMS AVE,YELLOW KNIFE,NT,None,4


Valores nulos por columna en los datos cargados:

In [3]:
df.isnull().sum()

unit_number          48798
street_number            0
street                   0
city                 15338
province                 0
postal_code          50000
__index_level_0__        0
dtype: int64

## 3. Transformación mínima: seleccionar columnas

Se seleccionan solo las columnas que describen la dirección:

- se descarta `__index_level_0__`, un índice técnico que quedó al guardar el dataset en parquet y no aporta información;
- se descarta `postal_code`, porque en las filas cargadas está vacía (100 % nulos).

La selección devuelve un **nuevo DataFrame**, que se guarda en `df_transformado`; ese es el que se pasa a la clase de resumen.

In [4]:
columnas = ["unit_number", "street_number", "street", "city", "province"]
df_transformado = df[columnas]

print("Antes:  ", df.shape)
print("Después:", df_transformado.shape)
df_transformado.head()

Antes:   (50000, 7)
Después: (50000, 5)


,unit_number,street_number,street,city,province
0,NaN,410,HALL CR,YELLOW KNIFE,NT
1,NaN,121,NIVEN DR,YELLOW KNIFE,NT
2,NaN,5608,51A AVE,YELLOW KNIFE,NT
3,NaN,233,BORDEN DR,YELLOW KNIFE,NT
4,NaN,604,WILLIAMS AVE,YELLOW KNIFE,NT


## 4. Generar el resumen en .txt con los datos transformados

In [5]:
summarizer = DatasetSummarizer(
    df_transformado,
    title="Resumen: direcciones de Canadá (datos transformados)",
)
ruta = summarizer.save("resumen_canadian_addresses.txt")
print("Archivo generado:", ruta)

Archivo generado: resumen_canadian_addresses.txt


## 5. Resultado: contenido del archivo .txt

In [6]:
print(ruta.read_text(encoding="utf-8"))

Resumen: direcciones de Canadá (datos transformados)

Cantidad de filas: 50000
Cantidad de columnas: 5

Nombres y tipos de columnas:
  - unit_number: str
  - street_number: str
  - street: str
  - city: str
  - province: str

Valores nulos por columna:
  - unit_number: 48798 (97.6 %)
  - street_number: 0 (0.0 %)
  - street: 0 (0.0 %)
  - city: 15338 (30.7 %)
  - province: 0 (0.0 %)
Total de valores nulos: 64136

Estadísticas descriptivas:
       unit_number street_number street          city province
count         1202         50000  50000         34662    50000
unique         289         13346   4073           449        3
top           #101            15  RTE 2  YELLOW KNIFE       PE
freq            54           321    274          6996    27666

Primeras 5 filas:
  unit_number street_number        street          city province
0         NaN           410       HALL CR  YELLOW KNIFE       NT
1         NaN           121      NIVEN DR  YELLOW KNIFE       NT
2         NaN          5608 

## Explicación de la transformación

Se aplicó una sola operación: **selección de columnas** con `df[columnas]`.

- El DataFrame pasó de 7 a 5 columnas y conservó las 50.000 filas.
- Se quitaron `__index_level_0__` (índice técnico sin significado) y `postal_code` (sin datos en esta muestra).
- El resultado se guardó en `df_transformado` y se pasó a `DatasetSummarizer`, por eso el archivo `resumen_canadian_addresses.txt` describe los datos transformados: muestra 5 columnas y ya no incluye las dos descartadas.
- En el resumen se observa que `unit_number` y `city` todavía tienen valores nulos, y que las filas cargadas corresponden a pocas provincias, porque el dataset viene ordenado por provincia y se tomaron las primeras filas.